In [4]:
import os
import streamlit as st
import pickle
import time
import langchain    
import openai
from dotenv import load_dotenv
load_dotenv()
from langchain import OpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.vectorstores import FAISS

In [5]:
loaders = UnstructuredURLLoader(
    urls=[
        "https://www.moneycontrol.com/news/india/modi-launches-bsnl-s-swadeshi-4g-network-from-odisha-commissions-97-500-towers-13585407.html",
        "https://www.moneycontrol.com/europe/?url=https://www.moneycontrol.com/technology/she-is-a-menace-to-donald-trump-wants-microsoft-to-fire-senior-executive-article-13585304.html",
    ]
)

data=loaders.load()
len(data)

2

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
doc=text_splitter.split_documents(data)
len(doc)

19

In [7]:
doc[3]

Document(metadata={'source': 'https://www.moneycontrol.com/news/india/modi-launches-bsnl-s-swadeshi-4g-network-from-odisha-commissions-97-500-towers-13585407.html'}, page_content='Helicopters on the Moon? NASA astronaut candidate’s skills could shape Lunar landings\n\nWorld Tourism Day: 9 Destinations That Look Too Beautiful to Be Real\n\nDurga Puja 2025 Dates: Full Schedule, Rituals, Start & End Time of the Grand Festival\n\nNothing CEO Carl Pei to invest $100M, create 1,800 jobs in India over next three years\n\nWorried about iPhone 17 Pro and iPhone 17’s scratchgate issue? Here’s what Apple has to say about it\n\n8 foods that damage your teeth and what to eat instead, according to a dentist\n\nBest exercises to burn belly fat and build stamina: Try these simple yoga poses and bodyweight workouts\n\nKareena Kapoor’s nutritionist Rujuta Diwekar shares 4 must-have Navratri foods\n\nAdvisory Alert:\n\nmoneycontrol\n\nFollow Us On:\n\nFacebook\n\ntwitter\n\ninstagram\n\nlinkedin\n\nteleg

In [8]:
import sys
!{sys.executable} -m pip list

Package                                  Version
---------------------------------------- ------------
absl-py                                  2.1.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.15
aiosignal                                1.4.0
altair                                   5.3.0
altgraph                                 0.17.4
annotated-types                          0.7.0
anyio                                    4.6.2.post1
artlearn                                 0.1.0
asttokens                                3.0.0
astunparse                               1.6.3
attrs                                    23.2.0
av                                       12.3.0
backoff                                  2.2.1
bcrypt                                   4.1.3
beautifulsoup4                           4.13.3
blinker                                  1.8.2
branca                                

In [9]:
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import os

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",google_api_key=os.getenv("GEMINI_API_KEY"))
vectorindex_gemini = Chroma.from_documents(documents=doc, embedding=embeddings)

In [11]:
# Assume 'doc' is a list of your loaded documents
# doc = [Document(page_content="..."), Document(page_content="...")] 
# --- EDITED PART ---
# 1. Define a directory name where the database will be stored.
#    Chroma saves a folder with multiple files, not a single .pkl file.
persist_directory = 'my_persistent_chroma_db'

# 2. Create the Chroma vector store and tell it to save to that directory.
#    The .from_documents method will create the vectors AND save them to disk.
vectorindex_gemini = Chroma.from_documents(
    documents=doc, 
    embedding=embeddings,
    persist_directory=persist_directory
)
print(f"Vector store has been created and saved to the directory: '{persist_directory}'")

Vector store has been created and saved to the directory: 'my_persistent_chroma_db'


In [12]:
# 1. Define your query
query = "Why US President Donald Trump has urged Microsoft ?"

# 2. Perform the search on your loaded vector store
#    This returns a list of the most relevant documents
relevant_docs = vectorindex_gemini.similarity_search(query)

# 3. Print the results
print(f"Found {len(relevant_docs)} relevant documents for the query: '{query}'\n")
for i, doc in enumerate(relevant_docs):
    print(f"--- Document {i+1} ---\n")
    print(doc.page_content)
    print("\n")

retriever = vectorindex_gemini.as_retriever()

Found 4 relevant documents for the query: 'Why US President Donald Trump has urged Microsoft ?'

--- Document 1 ---

Google BirthdayForza Horizon 6 Release dateSamsung Galaxy S26 UltraGoogle Pixel 10 reviewMeta Vibes feed

‘She is a menace to…', Donald Trump wants Microsoft to fire senior executive

US President Donald Trump has urged Microsoft to dismiss Lisa Monaco, a top executive and former deputy attorney general under Joe Biden, escalating his criticism of officials linked to past administrations.

MC Tech Desk

September 27, 2025 / 09:56 IST

Microsoft

Microsoft



Invite your friends and family to sign up for MC Tech 3, our daily newsletter that breaks down the biggest tech and startup stories of the day



DAILY-EVENING



SUBSCRIBE

End your day with a breakdown of the biggest tech and startup stories in your inbox



DAILY-EVENING



SUBSCRIBE

End your day with a breakdown of the biggest tech and startup stories in your inbox


--- Document 2 ---

Google BirthdayForza Hori

In [13]:
retriever = vectorindex_gemini.as_retriever()

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7, google_api_key=os.getenv("GEMINI_API_KEY"))
    template = """
    Answer the question based only on the following context.
    Cite the source from the metadata if available.

    Context:
    {context}

    Question: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(f"Content: {doc.page_content}\nSource: {doc.metadata.get('source', 'N/A')}" for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # --- Run the chain ---
    question = "Why US President Donald Trump has urged Microsoft ?"
    result = chain.invoke(question)

    langchain.debug=True

print(result[0])

IndentationError: unexpected indent (4262751453.py, line 6)